In [13]:
import os
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import tensorflow as tf
#libraries for building the model
from tensorflow.keras.layers import BatchNormalization, Conv2D,MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping




# Converting all the audio files into the .wav format
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment # will convert the format

from tensorflow.image import resize

import subprocess

import sys
from spleeter.separator import Separator

## Spleeter Separate

In [14]:
output_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output" 
input_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\genrenew\bollypop\bp03.mp3"

In [20]:
import os
from spleeter.separator import Separator

def spleeter_conversion(input_audio, test_output):   
    try:

        separator = Separator('spleeter:4stems')

        separator.separate_to_file(input_audio, test_output)


                # --- STEP 2: THE FIX ---
        # This wipes the memory and unlocks the "Graph"
        # tf.keras.backend.clear_session() used another cell

        # --- STEP 3: Now run Chunking and Load Model ---
        # chunks = your_chunking_function(audio_path)
        # model = tf.keras.models.load_model('my_model.keras')
        
        print("✅ Separation Complete!")

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        # Common error: If your GPU memory is full, try restarting the kernel

In [21]:
# will finalize the graph and it can't be changed so need to remove this process graph resources after usage
# second time run this cell , nesting violated error, but same graph by the spleeter so run , but tenserflow diff(version graph) can't run
ffmpeg_bin = r'C:\ffmpeg\bin'
os.environ["PATH"] = ffmpeg_bin + os.pathsep + os.environ["PATH"]
spleeter_conversion(input_path, output_path)    

INFO:tensorflow:Using config: {'_model_dir': 'pretrained_models\\4stems', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': gpu_options {
  per_process_gpu_memory_fraction: 0.7
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Apply unet for vocals_spectrogram
INFO:tensorflow:Apply unet for drums_spectrogram
INFO:tensorflow:Apply unet

Exception ignored in: <generator object Estimator.predict at 0x00000174C1DEB060>
Traceback (most recent call last):
  File "c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\site-packages\tensorflow_estimator\python\estimator\estimator.py", line 618, in predict
    with tf.Graph().as_default() as g:
  File "c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\contextlib.py", line 153, in __exit__
    self.gen.throw(typ, value, traceback)
  File "c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\site-packages\tensorflow\python\framework\ops.py", line 5821, in get_controller
    with super(_DefaultGraphStack,
  File "c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\contextlib.py", line 153, in __exit__
    self.gen.throw(typ, value, traceback)
  File "c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\site-packages\tensorflow\python\framework\ops.py", l

INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from pretrained_models\4stems\model
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
✅ Separation Complete!


In [17]:
other_stem_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output\bp01\other.wav"

In [23]:
tf.keras.backend.clear_session()

AssertionError: Do not use tf.reset_default_graph() to clear nested graphs. If you need a cleared graph, exit the nesting and create a new graph.

In [24]:
#chunking part
def chunking(other_stem_path, sample_set_rate = 22050, target_shape = (150,150), threshold = 0.01 ):
        # --- STEP 2: Load and Chunk ---

    try:
        audio_data, sample_rate = librosa.load(other_stem_path, sr = sample_set_rate)
    except Exception as e:
        print(f"Skipping corrupted files: {other_stem_path}: {e}")
        
    #define the duration of each chunk and overlap
    chunk_duration = 4
    overlap_duration = 2
    data = []

    #convert duration to sample
    chunk_samples = int(chunk_duration*sample_rate)
    stride = int((chunk_duration-overlap_duration)*sample_rate) ## movement(4-2) * sr -> 2 second movement
    
    # sliding window logic
    # Use range with stride to ensure 'start + chunk_samples' never exceeds len(y)
    for start in range(0, len(audio_data) - chunk_samples + 1, stride):
        end = start + chunk_samples
        chunk = audio_data[start:end]

        # QUALITY CHECK: Check if chunk has actual sound (RMS energy)
        rms = np.sqrt(np.mean(chunk**2))
        if rms > threshold: 

            # melspectrogram part, this is the matrix we are getting the 
            mel_spectrogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate, n_mels = 150) #calculating spectrogram by this
            mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref= np.max)


            # 2. APPLY NORMALIZATION (Min-Max Scaling)
            # This ensures the test data matches the 0-1 range of your training data
            mel_min = mel_spectrogram.min()
            mel_max = mel_spectrogram.max()
            if mel_max - mel_min > 0:
                mel_norm = (mel_spectrogram - mel_min) / (mel_max - mel_min)
            else:
                mel_norm = mel_spectrogram # Handle silence/constant signal

            mel_spectrogram = resize(np.expand_dims(mel_norm, axis = -1), target_shape).numpy()

            # appending the data to the lists
            data.append(mel_spectrogram)
                
    return np.array(data)
    

In [25]:
X_test = chunking(other_stem_path=other_stem_path)

RuntimeError: Graph is finalized and cannot be modified.

In [ ]:
graph finalized error 